In [4]:
import pandas as pd
import numpy as np

In [28]:
import string
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

In [5]:
tweets = pd.read_csv('tweets-data.csv', encoding = 'latin-1', header = None)

In [6]:
tweets.head()

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [7]:
tweets.tail()

,0,1,2,3,4,5
1599995,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1599996,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...
1599999,4,2193602129,Tue Jun 16 08:40:50 PDT 2009,NO_QUERY,RyanTrevMorris,happy #charitytuesday @theNSPCC @SparksCharity...


In [8]:
tweets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 6 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   0       1600000 non-null  int64 
 1   1       1600000 non-null  int64 
 2   2       1600000 non-null  object
 3   3       1600000 non-null  object
 4   4       1600000 non-null  object
 5   5       1600000 non-null  object
dtypes: int64(2), object(4)
memory usage: 73.2+ MB


In [9]:
tweets.rename(columns = {0:'target',
                        1:'ids',
                        2:'date',
                        3:'flag',
                        4:'user',
                        5:'text'}, inplace = True)

In [10]:
tweets

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
...,...,...,...,...,...,...
1599995,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1599996,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...


In [11]:
tweets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 6 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   target  1600000 non-null  int64 
 1   ids     1600000 non-null  int64 
 2   date    1600000 non-null  object
 3   flag    1600000 non-null  object
 4   user    1600000 non-null  object
 5   text    1600000 non-null  object
dtypes: int64(2), object(4)
memory usage: 73.2+ MB


## Drop irrelevant columns

In [12]:
col = ['ids', 'date', 'flag', 'user']
tweets.drop(columns = col, inplace = True)

In [13]:
tweets.sample(5)

,target,text
727349,0,"@pfeifnugget Haha, sounds good. I need a repla..."
151217,0,@DevineNews aww hope he gets better. I've alw...
1250055,4,@patrickdawgg thank u for both
1348891,4,Eating cold stone. with rett
1362640,4,yess for @jeremyg423 new tank.. can't wait to ...


In [14]:
df = tweets.copy()

## Assign polarity of tweets
(0 = negative, 4 = positive)

In [15]:
def polarity(target):
    if target == 0:
        return 'Negative'
    else:
        return 'Positive'

In [16]:
df['target'] = df['target'].apply(polarity)

In [17]:
df['target'].value_counts()

target
Negative    800000
Positive    800000
Name: count, dtype: int64

(negative = 0, positive = 1)

In [18]:
def label(target):
    if target == 'Negative':
        return 0
    else:
        return 1

In [19]:
df['target'] = df['target'].apply(label)

In [20]:
df['target'].value_counts()

target
0    800000
1    800000
Name: count, dtype: int64

## check for duplicate col

In [21]:
df.duplicated().sum()

np.int64(16309)

## drop duplicated col

In [22]:
df.drop_duplicates(inplace = True)

In [23]:
df.duplicated().sum()

np.int64(0)

## Preprocessing
- Lower case
- Tokenization
- Removing special characters
- Removing stop words and punctuation
- Stemming

In [29]:
def transform_txt(txt):
    # Remove URLs 
    txt = re.sub(r'http\S+|www\S+|https\S+', '', txt, flags = re.MULTILINE)
    
    # Remove @mentions but keep hashtag content
    txt = re.sub(r'@\w+', '', txt)
    txt = re.sub(r'#', '', txt)
    
    tokens = re.findall(r'\b\w+\b', txt.lower()) # tokenize
    result = [ps.stem(token) for token in tokens if token not in stop_words] #stem
    
    return " ".join(result)

In [30]:
df['processed_text'] = df['text'].apply(transform_txt)

In [37]:
df.sample(5)

,target,text,processed_text
1152929,1,@brighters Presume you meant: &quot;T'sun'll b...,presum meant quot sun bi crackin flag quot
772387,0,boohoo lost my shirt in Vegas,boohoo lost shirt vega
760591,0,Is losing faith that my Tegan and Sara hoodie ...,lose faith tegan sara hoodi shirt ever ship li...
146181,0,I hate school .GiveMeSexy.&lt;3,hate school givemesexi lt 3
1075070,1,@dannywood U should just stay in LA!,u stay la


## export processed data

In [39]:
df.to_csv('processed-tweets.csv', index = False)